In [1]:
import pandas as pd
from collections import defaultdict
from tqdm import tqdm
import math

In [7]:
import os

# Colab 上先挂载 Drive（已挂载则跳过；Kaggle 上没有 google.colab 模块）
if not os.path.isdir('/content/drive/MyDrive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        pass

DATA = ('/content/drive/MyDrive/dataset/TianchiNewsRec/'
        if os.path.isdir('/content/drive/MyDrive/dataset/TianchiNewsRec')
        else '/kaggle/input/datasets/jiaweiwei0311/tianchinewsrec/')

train_click = pd.read_csv(DATA + 'train_click_log.csv')
testA_click = pd.read_csv(DATA + 'testA_click_log.csv')
articles = pd.read_csv(DATA + 'articles.csv').rename(columns={"article_id":"click_article_id"})
all_click = pd.concat([train_click,testA_click], ignore_index=True)
train_click.columns.tolist(),testA_click.columns.tolist(),articles.columns.tolist()

(['user_id',
  'click_article_id',
  'click_timestamp',
  'click_environment',
  'click_deviceGroup',
  'click_os',
  'click_country',
  'click_region',
  'click_referrer_type'],
 ['user_id',
  'click_article_id',
  'click_timestamp',
  'click_environment',
  'click_deviceGroup',
  'click_os',
  'click_country',
  'click_region',
  'click_referrer_type'],
 ['click_article_id', 'category_id', 'created_at_ts', 'words_count'])

In [14]:
# ============================================================
# 环境 bootstrap：让项目可 import（Colab/Drive 或 Kaggle）
#   Colab ：优先用 Drive 上的持久化副本，没有就 clone 进去（下次不用再上传）
#   Kaggle：clone 到 /kaggle/working/news_recommender
# ============================================================
import os, sys, subprocess

REPO_URL = 'https://github.com/hydrofluoric07/news_recommender.git'
DRIVE_PROJ = '/content/drive/MyDrive/codes/TianchiNewsRec'
KAGGLE_PROJ = '/kaggle/working/news_recommender'
UPDATE = False          # 本地 push 了新代码后设成 True，跑一次拉最新版

# 1. Colab：确保 Drive 已挂载（cell 2 通常已经挂过，这里是兜底）
if not os.path.isdir('/content/drive/MyDrive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        pass

# 2. 选项目目录
if os.path.isdir('/content/drive/MyDrive'):
    PROJ = DRIVE_PROJ                     # Colab + Drive（持久化）
elif os.path.isdir('/kaggle/working'):
    PROJ = KAGGLE_PROJ                    # Kaggle
else:
    PROJ = os.path.join(os.getcwd(), 'news_recommender')

# 3. 没有就 clone；已在 Drive 里就跳过（这就是"只上传一次"的关键）
if not os.path.exists(os.path.join(PROJ, 'main.py')):
    os.makedirs(os.path.dirname(PROJ), exist_ok=True)
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, PROJ],
                   check=True, timeout=300,
                   env=dict(os.environ, GIT_TERMINAL_PROMPT='0'))
    print('已 clone 到', PROJ)
elif UPDATE and os.path.isdir(os.path.join(PROJ, '.git')):
    subprocess.run(['git', '-C', PROJ, 'pull', '--ff-only'],
                   check=True, timeout=300,
                   env=dict(os.environ, GIT_TERMINAL_PROMPT='0'))
    print('已更新', PROJ)
elif UPDATE:
    print('⚠️ 该目录不是 git 仓库，跳过 pull（需要手动同步代码）')

# 4. 数据 / 缓存 / 输出都指向 Drive，main.py 不用改代码
if os.path.isdir('/content/drive/MyDrive'):
    os.environ.setdefault('NEWS_DATA_PATH',
                          '/content/drive/MyDrive/dataset/TianchiNewsRec/')
    os.environ.setdefault('NEWS_CACHE_DIR', os.path.join(PROJ, 'cache'))
    os.environ.setdefault('NEWS_OUTPUT_DIR', os.path.join(PROJ, 'results'))

# 5. kernel 里的 import 和之后的 !python main.py 都用这个目录
if PROJ not in sys.path:
    sys.path.insert(0, PROJ)
os.chdir(PROJ)
print('项目目录:', PROJ)
print('数据目录:', os.environ.get('NEWS_DATA_PATH', '（未设置，走默认解析）'))

项目目录: /content/drive/MyDrive/codes/TianchiNewsRec
数据目录: /content/drive/MyDrive/dataset/TianchiNewsRec/


In [15]:
# ============================================================
# （可选）改完项目里的 .py 后，运行这一格让 kernel 用上新代码
#
# 为什么删掉了 %load_ext autoreload：
#   这个镜像的 IPython 在 autoreload.py 里还写着 `from imp import reload`，
#   而 imp 模块在 Python 3.12 就被移除了（报错里的 python3.13），
#   所以必然 ModuleNotFoundError: No module named 'imp'。
#   不升级 IPython（会影响 Colab 环境），改成显式 reload。
# ============================================================
import sys, importlib

PROJECT_MODULES = (
    'main', 'utils', 'data_processing', 'submission', 'online',
    'offline_evaluation_multi',
    'models.recall_baseline', 'models.itemcf_recall',
    'models.usercf_recall', 'models.YouTubeDNN_torch',
)

def reload_project():
    loaded = []
    for name in PROJECT_MODULES:
        if name in sys.modules:
            importlib.reload(sys.modules[name])
            loaded.append(name)
    print('已重载:', loaded or '（还没有 import 过项目模块）')

print('项目路径:', sys.path[0])

项目路径: /content/drive/MyDrive/codes/TianchiNewsRec


In [19]:
from collections import Counter
gt = dict(zip(val_df.user_id, val_df.click_article_id))

ranks = []
for u, lst in recall_off.items():
    ids = [i for i, _ in lst[:10]]
    ranks.append(ids.index(gt[u]) + 1 if gt[u] in ids else 0)

print('名次分布(0=未进前10):', dict(sorted(Counter(ranks).items())))
print('Recall@5 =', sum(1 for r in ranks if 1 <= r <= 5) / len(ranks))
print('MRR@10   =', sum(1 / r for r in ranks if r) / len(ranks))

hot10 = list(hot_tr[:10])
print('只推热门Top10 的 Recall@10 =', sum(1 for u in recall_off if gt.get(u) in hot10) / len(recall_off))
print('冷启动用户数(无历史) =', sum(1 for u in recall_off if not uit_tr.get(u)))


NameError: name 'recall_off' is not defined

In [ ]:
# ============================================================
# 对照实验：无加权 ItemCF vs 加权 ItemCF
# 同一份切分、同一批用户、同一个指标函数，只换模型
# 依赖上面的 bootstrap 单元格；相似度复用前面算好的缓存
# ============================================================
import os, time
import pandas as pd

from data_processing import get_all_click_df, get_item_info_df, resolve_data_path
from utils import get_user_item_time_dict, get_item_topk_click, get_item_info_dict
from models.recall_baseline import itemcf_sim as itemcf_sim_base, generate_user_recall_dict
from models.itemcf_recall import itemcf_sim as itemcf_sim_w, generate_itemcf_recall_dict

data_path = resolve_data_path()
cache_dir = os.environ.get('NEWS_CACHE_DIR', './cache/')

# ---- 1. 切分（和线下评估完全一致）----
off = get_all_click_df(data_path=data_path, offline=True) \
        .sort_values(['user_id', 'click_timestamp'])
val_df = off.groupby('user_id').tail(1)
trn_df = off.drop(val_df.index)
gt = dict(zip(val_df['user_id'], val_df['click_article_id']))

uit_tr = get_user_item_time_dict(trn_df)
hot_tr = get_item_topk_click(trn_df, k=50)
_, _, item_created_time_dict = get_item_info_dict(get_item_info_df(data_path))

# ---- 2. 相似度：复用线下评估 / 加权单元格已经算好的缓存 ----
t0 = time.time()
i2i_base = itemcf_sim_base(trn_df,
                           save_path=os.path.join(cache_dir, 'itemcf_sim_offline.pkl'),
                           use_cache=True)
i2i_w = itemcf_sim_w(trn_df, item_created_time_dict,
                     os.path.join(cache_dir, 'itemcf_w_offline.pkl'),
                     use_cache=True)
print(f'相似度加载/计算完成，用时 {time.time() - t0:.0f}s')

def top5(d):
    return [i for i, _ in sorted(d.items(), key=lambda x: -x[1])[:5]]

sample_items = list(i2i_base)[:500]
changed = sum(top5(i2i_base[i]) != top5(i2i_w[i]) for i in sample_items)
print(f'抽样 {len(sample_items)} 个物品，加权后 top-5 邻居变化: {changed}')

# ---- 3. 同一批用户（先 1 万；跑通后可以调大）----
N_USERS = 10000
users_eval = pd.DataFrame({'user_id': val_df['user_id'].unique()[:N_USERS]})
for u in users_eval['user_id']:
    uit_tr.setdefault(u, [])
print(f'评估用户 {len(users_eval)}')

# ---- 4. 两个模型，用同一份历史 ----
t0 = time.time()
recall_base = generate_user_recall_dict(
    val_df=users_eval, user_item_time_dict=uit_tr, i2i_sim=i2i_base,
    sim_item_topk=10, recall_item_num=10, item_topk_click=hot_tr,
    save_path=os.path.join(cache_dir, 'recall_base_cmp.pkl'), use_cache=False)
print(f'baseline 召回用时 {time.time() - t0:.0f}s')

t0 = time.time()
recall_w = generate_itemcf_recall_dict(
    val_df=users_eval, user_item_time_dict=uit_tr, i2i_sim=i2i_w,
    sim_item_topk=10, recall_item_num=10, item_topk_click=hot_tr,
    item_created_time_dict=item_created_time_dict,
    emb_i2i_sim={},                      # 先关掉内容相似度，只比加权本身
    save_path=os.path.join(cache_dir, 'recall_w_cmp.pkl'), use_cache=False)
print(f'weighted 召回用时 {time.time() - t0:.0f}s')

# ---- 5. 同一套指标 ----
def eval_recall(recall, gt, k_recall=10, k_mrr=5):
    ranks = []
    for u, lst in recall.items():
        ids = [i for i, _ in lst[:k_recall]]
        ranks.append(ids.index(gt[u]) + 1 if u in gt and gt[u] in ids else 0)
    n = len(ranks)
    return (sum(1 for r in ranks if 1 <= r <= k_recall) / n,
            sum(1 / r for r in ranks if 1 <= r <= k_mrr) / n)

print('\n模型        Recall@10   MRR@5')
for name, rec in [('baseline', recall_base), ('weighted', recall_w)]:
    r10, m5 = eval_recall(rec, gt)
    print(f'{name:10s}  {r10:.4f}      {m5:.4f}')

same = sum([i for i, _ in recall_base[u][:5]] == [i for i, _ in recall_w[u][:5]]
           for u in recall_base)
print(f'\ntop-5 完全相同的用户: {same}/{len(recall_base)} ({same / len(recall_base):.2%})')

In [ ]:
!python main.py

python3: can't open file '/kaggle/working/main.py': [Errno 2] No such file or directory
